# Exercise 3: Morphological ambiguity

*Morphological Structure · version 0.1.0*

> List all readings the Turkish transducer returns for *yazar*, *çalışır* and *evleri* (use the readings call of the Hands-On section, or Zemberek). For each word: (a) gloss each reading; (b) write two sentences in which different readings are correct; (c) say which reading the pipeline selects in each sentence, and whether it is right.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0"`. Where the compiled `hfst` package has no wheel for your Python version, install `turkicnlp==0.5.0 pyhfst` instead: the first code cell then uses the pure-Python `pyhfst` reader, which has the same lookup interface. The Apertium transducers are downloaded once with `turkicnlp.download`. With `pyhfst`, readings may be listed in a different order than with `hfst`, and the pipeline may select a different reading when several have the same weight.

The first cell lists the readings without `+` continuations (the copula readings such as `+i<cop><aor><p3><sg>` add 'is …' to every nominal reading). The second runs the pipeline on two sentences per word, chosen so that different readings are correct.

In [ ]:
import sys
try:
    import hfst                                   # compiled HFST, as in the chapter
except ImportError:                               # fall back to the pure-Python reader
    import pyhfst
    sys.modules["hfst"] = pyhfst

import turkicnlp
from turkicnlp import Pipeline

def pipeline(lang):
    turkicnlp.download(lang, processors=["morph"])          # Apertium transducers (once)
    return Pipeline(lang, processors=["tokenize", "morph"], tokenize_backend="rule", morph_backend="apertium")

def analyser(pipe):
    """The Apertium processor of a pipeline (private API in 0.5.0)."""
    return next(p for p in pipe._processors if hasattr(p, "_analyze"))

def readings(pipe, word, plus=False):
    """All transducer readings of one word; by default without '+' continuations (copula etc.)."""
    out = [r["raw"] for r in analyser(pipe)._analyze(word)]
    return out if plus else [r for r in out if "+" not in r]

def show(pipe, sentence):
    for w in pipe(sentence).words:
        print(f"  {w.text:<18} lemma={w.lemma or '_':<10} upos={w.upos or '_':<6} feats={w.feats or '_'}")

tur = pipeline("tur")

In [ ]:
for word in ["yazar", "çalışır", "evleri"]:
    print(word)
    for r in readings(tur, word):
        print("   ", r)

SENTENCES = {
    "yazar":   ["Yazar yeni bir kitap yazdı.", "Ali her gün mektup yazar."],
    "çalışır": ["Ali her gün çalışır.", "Makine çalışır durumda."],
    "evleri":  ["Onların evleri çok büyük.", "Ali bütün evleri boyadı."],
}
for word, sentences in SENTENCES.items():
    for s in sentences:
        w = next(x for x in tur(s).words if x.text.lower() == word)
        print(f"{s:30} -> {w.text}: lemma={w.lemma} upos={w.upos} feats={w.feats}")

## Worked solution

Output of `experiments/ch05_exercises/run_ch05_exercises.py` on 19 October 2026 (TurkicNLP 0.5.0, HFST 3.16.0.1, Python 3.11):

```
yazar
    yaz<v><tv><aor>
    yaz<v><tv><ger_aor><nom>
    yaz<v><tv><gna_aor>
    yaz<v><tv><gpr_aor>
    yaz<v><tv><gpr_aor><subst><nom>
    yazar<n><attr>
    yazar<n><nom>
çalışır
    çal<v><tv><coop><aor>
    çal<v><tv><coop><ger_aor><nom>
    çal<v><tv><coop><gna_aor>
    çal<v><tv><coop><gpr_aor>
    çal<v><tv><coop><gpr_aor><subst><nom>
    çalış<v><tv><aor>
    çalış<v><tv><ger_aor><nom>
    çalış<v><tv><gna_aor>
    çalış<v><tv><gpr_aor>
    çalış<v><tv><gpr_aor><subst><nom>
evleri
    ev<n><pl><acc>
    ev<n><pl><px3sp><nom>
    ev<n><px3pl><nom>
Yazar yeni bir kitap yazdı.    -> Yazar: lemma=yaz upos=VERB feats=_
Ali her gün mektup yazar.      -> yazar: lemma=yaz upos=VERB feats=Aspect=Hab|Tense=Pres
Ali her gün çalışır.           -> çalışır: lemma=çalış upos=VERB feats=Aspect=Hab|Tense=Pres
Makine çalışır durumda.        -> çalışır: lemma=çalış upos=VERB feats=_
Onların evleri çok büyük.      -> evleri: lemma=ev upos=NOUN feats=Case=Acc|Number=Plur
Ali bütün evleri boyadı.       -> evleri: lemma=ev upos=NOUN feats=Case=Acc|Number=Plur
```

**(a) Glosses.** *yazar*: the aorist of *yaz-* 'write', 3SG 'writes' (`<aor>`); the same form used as a participle 'writing (one)' (`gpr_aor`), as a nominal (`ger_aor`, `<subst>`) or a converb (`gna_aor`, as in *yazar yazmaz* 'as soon as X writes'); and the noun *yazar* 'author' (`<n><nom>`, or `<attr>` before a noun). *çalışır*: the aorist of *çalış-* 'work', with the same participle, nominal and converb uses; the transducer also builds *çal-ış-* from *çal-* 'play, steal' with the cooperative suffix (`coop`), a historically real but here implausible analysis. *evleri*: `px3pl` 'their house(s)'; `pl` + `px3sp` 'his/her houses' or 'their houses'; `pl` + `acc` 'the houses' (definite object).

**(b, c) Sentences and the pipeline's choice.**

| Sentence | Correct reading | Selected | Right? |
|---|---|---|---|
| *Yazar yeni bir kitap yazdı.* 'The author wrote a new book.' | noun *yazar* | verb *yaz-* | no |
| *Ali her gün mektup yazar.* 'Ali writes a letter every day.' | aorist 3SG | verb, `Aspect=Hab\|Tense=Pres` | yes |
| *Ali her gün çalışır.* 'Ali works every day.' | aorist 3SG | verb, `Aspect=Hab\|Tense=Pres` | yes |
| *Makine çalışır durumda.* 'The machine is in working order.' | aorist participle | verb, no features | partly: no `VerbForm=Part` |
| *Onların evleri çok büyük.* 'Their houses are very big.' | PL + POSS.3 (or POSS.3PL) | PL + ACC | no |
| *Ali bütün evleri boyadı.* 'Ali painted all the houses.' | PL + ACC | PL + ACC | yes |

The selection is made by weights and heuristics, not by a model of the sentence: the noun reading of *yazar* is not chosen at the start of a sentence before *yeni bir kitap*, and *evleri* gets the same accusative reading after the genitive *onların*, which calls for a possessive, as after *bütün* before a transitive verb. All readings have weight 0 in this transducer, so the choice also depends on the order in which the transducer lists them. With the pure-Python `pyhfst` reader, which lists them in a different order, both *evleri* sentences got POSS.3PL instead, so the right and wrong sentences swap. Six sentences show the kinds of errors; they are not an accuracy estimate.

## The recurring sentence

The same two steps for the book's recurring Kazakh sentence: all transducer readings of each word, then the pipeline's selection. Compare them with the glosses in the chapter's section on the recurring examples.

In [ ]:
kaz = pipeline("kaz")
sentence = "Қазақстан осы өңірдегі бейбітшілікті қолдайды."   # the recurring Kazakh sentence
for word in sentence.rstrip(".").split():
    print(word, readings(kaz, word))
show(kaz, sentence)

Recorded output (20 October 2026; TurkicNLP 0.5.0, compiled HFST):

```
Қазақстан ['Қазақстан<np><top><attr>', 'Қазақстан<np><top><nom>']
осы ['осы<det><dem>', 'осы<prn><dem><nom>']
өңірдегі ['өңір<n><loc><attr>', 'өңір<n><loc><subst><nom>']
бейбітшілікті ['бейбітшілік<n><acc>']
қолдайды ['қол<adj><subst><sim><acc>', 'қол<n><sim><acc>', 'қолда<v><tv><aor><p3><pl>', 'қолда<v><tv><aor><p3><sg>']
  Қазақстан          lemma=Қазақстан  upos=PROPN  feats=_
  осы                lemma=осы        upos=PRON   feats=Case=Nom|PronType=Dem
  өңірдегі           lemma=өңір       upos=NOUN   feats=Case=Loc
  бейбітшілікті      lemma=бейбітшілік upos=NOUN   feats=_
  қолдайды           lemma=қолда      upos=VERB   feats=Aspect=Hab|Number=Plur|Person=3|Tense=Pres
  .                  lemma=.          upos=PUNCT  feats=_
```

The transducer has the analyses of the chapter's gloss: *өңір<loc><attr>*, *бейбітшілік<acc>* and *қолда<aor><p3>*, the last with both a singular and a plural reading, because the Kazakh 3rd-person ending does not mark number. The pipeline's selection loses some of this: *осы* becomes a pronoun instead of a determiner, the accusative of *бейбітшілікті* is dropped from the features, and *қолдайды* is given `Number=Plur` for a singular subject. The transducer also offers *қол-дай* 'like a hand' (similative) readings, which the context rules out.

## Your turn — extension

1. Add *okuma* (verbal noun 'reading', negative imperative 'don't read!', and *ok-um-a* 'to my arrow') and write a sentence for each reading.
2. Count, for your six sentences, how often the selected reading is right. Six sentences are a check of your understanding, not an accuracy estimate.